# B（水果專用＋新增 5 類）：21 類水果 ＋ 奇異果、覆盆子、黑莓、哈密瓜、菠蘿蜜 ＝ 26 類（Colab）
> **版本 v8**　接在 `Colab_B_retrain.ipynb`（v5）之後。原本的 v5 與已存的權重都不會被動到。

**先把執行階段改成 GPU**（執行階段 → 變更執行階段類型 → T4 GPU），再一格一格往下執行。

**做了什麼**
1. 步驟 5（B0f）：拿掉 9 類蔬菜，剩 21 類水果。
2. 步驟 6（新）：把你放在 Drive `fruit_AB_out/extra/` 的資料解壓縮，用 B0h 加進去：
   LVIS（奇異果、覆盆子、黑莓、哈密瓜、香蕉、桃子、櫻桃）、data_all（香蕉、黑莓）、Jackfruit（菠蘿蜜）、cantaloupe（哈密瓜）。共 26 類。
3. 訓練設定和 v5 相同（yolov8s、50 個 epoch、批次 16、640 px），權重另存成 `b_yolo_fruit26.pt`，**不會覆蓋**舊的。

**不能做的事**：`stock_test_set`、`real_test109` 的照片**不可以**放進訓練資料（它們是我們另外用來比較新舊模型的測試照片）。

**驗收標準（先講好）**：訓練完用同樣的 189 張照片、R0 設定重跑：(1) 原本 21 類水果「種類對的框」不比舊模型少超過 5%；(2) 新增的 5 類至少有幾類開始被框到並叫對；(3) 香蕉、桃子沒有變差。沒有達到就繼續用舊模型加 R9。

**要有的預期**：新增類別的資料很少，場景也和我們的測試照片不同，可能改善有限。這是一個可驗證的小實驗，不保證變準。

**訓練時間**：和 v5 差不多，T4 上可能要好幾個小時（我沒有實測）。想先試跑，把步驟 7 的 `--epochs 50` 改成 `--epochs 1`。

**步驟 1　連接 Google Drive。** 會跳出授權視窗，按允許。

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


**步驟 2　解壓縮專案並安裝套件。**（和 v5 相同）

In [2]:
!cp /content/drive/MyDrive/fruit_AB.zip /content/
!unzip -q -o /content/fruit_AB.zip -d /content/
%cd /content/fruit_AB
!pip install -q ultralytics pyyaml

/content/fruit_AB


## 步驟 3　下載資料集（和 v5 相同）
Zenodo 18904638 的 `AgricultureDataset.zip`（約 959 MB，Colab 約需 16 分鐘；看到 `saved` 就是成功）。
請在使用時標註出處：George Bayón, Mencía；Sánchez González, Lidia；Universidad de León；DOI 10.5281/zenodo.18904638；CC BY 4.0。

In [3]:
!wget -O /content/dfv16.zip "https://zenodo.org/records/18904638/files/AgricultureDataset.zip?download=1"
!mkdir -p /content/dfv16
!unzip -q -o /content/dfv16.zip -d /content/dfv16

--2026-11-04 16:16:12--  https://zenodo.org/records/18904638/files/AgricultureDataset.zip?download=1
Resolving zenodo.org (zenodo.org)... 188.185.45.63, 137.138.121.78, 188.185.40.132, ...
Connecting to zenodo.org (zenodo.org)|188.185.45.63|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1005423492 (959M) [application/octet-stream]
Saving to: ‘/content/dfv16.zip’

/content/dfv16.zip  100%[===================>] 958.85M  1.72MB/s    in 8m 19s  

2026-11-04 16:24:32 (1.92 MB/s) - ‘/content/dfv16.zip’ saved [1005423492/1005423492]



In [4]:
!unzip -p /content/dfv16.zip "*data.yaml"

train: ../train/images
val: ../valid/images
test: ../test/images

nc: 30
names: ['Apple', 'Apricot', 'Aubegine', 'Banana', 'Bean', 'Carrot', 'Cherry', 'Courgette', 'DragonFruit', 'Durian', 'Fig', 'Grape', 'Lemon', 'Lime', 'Lychee', 'Mango', 'Orange', 'Peach', 'Pear', 'Pepper', 'Pineapple', 'Plum', 'Pomegranate', 'Quince', 'Radish', 'Rice', 'Salad', 'Strawberry', 'Tomato', 'Watermelon']

roboflow:
  workspace: vision-rzoan
  project: datasettfg-2502
  version: 2
  license: CC BY 4.0
  url: https://universe.roboflow.com/vision-rzoan/datasettfg-2502/dataset/2

## 步驟 4　記下資料集位置
這一格會產生 `/content/b_root.txt`（步驟 5 要用）。**輸出要看到 train、valid、test 三個資料夾。**

In [5]:
!dirname "$(find /content/dfv16 -name data.yaml | head -n 1)" > /content/b_root.txt
!echo "資料集位置：" && cat /content/b_root.txt
!ls "$(cat /content/b_root.txt)"

資料集位置：
/content/dfv16/AgricultureDataset
data.yaml  README.dataset.txt  README.roboflow.txt  test  train  valid


## 步驟 5　B0f：做「只有水果」的資料集（21 類）
**5-1** 把轉換程式存到 `B/B0f_make_fruit_only.py`（已經包在這個 Notebook 裡）。**5-2** 執行；輸出要顯示保留 **21 類**、拿掉 **9 類**。
重跑時會先刪掉舊的 `/content/dfv_fruit`，避免舊檔案混進來。

In [6]:
%%writefile B/B0f_make_fruit_only.py
# -*- coding: utf-8 -*-
"""B0f：把資料集做成「只有水果」的版本（拿掉蔬菜等 9 個非水果類別）。
做的事（依「類別名稱」判斷，不依編號，所以不用事先知道 data.yaml 的編號順序）：
  1. 讀資料集原本的 data.yaml，找出類別名稱。
  2. 把 9 個非水果類別的框從標註檔刪掉，其餘水果類別依原順序重新編號（0, 1, 2 ...）。
  3. 一張圖刪完框後如果一個框都不剩（原本只有蔬菜），整張圖不放進新資料集。
  4. 輸出到新資料夾（圖片用硬連結，不佔空間；不行就複製）。不會改動原本的資料集。
  5. 產生新的 data_fruit.yaml 與類別清單，並印出每個資料組刪了多少圖、多少框。
用法：python B/B0f_make_fruit_only.py --root /content/dfv16 --out /content/dfv_fruit
"""
import argparse
import glob
import os
import shutil
import sys

import yaml

NON_FRUIT = {"aubegine", "bean", "carrot", "courgette", "pepper", "radish", "rice", "salad", "tomato"}
SPLITS = [("train", ["train"]), ("val", ["val", "valid", "validation"]), ("test", ["test"])]
IMG_EXT = (".jpg", ".jpeg", ".png", ".bmp", ".JPG", ".JPEG", ".PNG")


def find_yaml(root):
    found = []
    for path in glob.glob(os.path.join(root, "**", "*.yaml"), recursive=True):
        try:
            data = yaml.safe_load(open(path, encoding="utf-8"))
        except Exception:
            continue
        if isinstance(data, dict) and "names" in data:
            found.append(path)
    if not found:
        raise SystemExit("找不到含 names 的 data.yaml：" + root)
    found.sort(key=lambda p: (len(p), p))
    return found[0]


def load_names(data):
    names = data["names"]
    if isinstance(names, dict):
        return {int(k): str(v) for k, v in names.items()}
    return {i: str(v) for i, v in enumerate(names)}


def find_image_dir(root, aliases):
    """找某個資料組的圖片資料夾：支援 root/train/images 和 root/images/train 兩種排法。"""
    for alias in aliases:
        for candidate in (os.path.join(root, alias, "images"), os.path.join(root, "images", alias)):
            if os.path.isdir(candidate):
                return candidate
    return None


def label_dir_of(image_dir):
    parts = image_dir.replace("\\", "/").split("/")
    for i in range(len(parts) - 1, -1, -1):
        if parts[i] == "images":
            parts[i] = "labels"
            return "/".join(parts)
    raise SystemExit("圖片資料夾路徑裡找不到 images：" + image_dir)


def place(src, dst):
    if os.path.exists(dst):
        return
    try:
        os.link(src, dst)
    except Exception:
        shutil.copy2(src, dst)


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--root", required=True, help="資料集資料夾（解壓縮後）")
    parser.add_argument("--out", required=True, help="新的水果專用資料集要放的資料夾")
    parser.add_argument("--names_out", default="", help="另存類別清單（txt）")
    args = parser.parse_args()

    yaml_path = find_yaml(args.root)
    data = yaml.safe_load(open(yaml_path, encoding="utf-8"))
    old_names = load_names(data)
    print("資料集 data.yaml：", yaml_path)
    print("原本", len(old_names), "類：", [old_names[i] for i in sorted(old_names)])

    drop_ids = {i for i, n in old_names.items() if n.strip().lower() in NON_FRUIT}
    keep_ids = [i for i in sorted(old_names) if i not in drop_ids]
    new_id = {old: new for new, old in enumerate(keep_ids)}
    new_names = [old_names[i] for i in keep_ids]
    missing = sorted(NON_FRUIT - {old_names[i].strip().lower() for i in drop_ids})
    print("要拿掉的類別（", len(drop_ids), "）：", [old_names[i] for i in sorted(drop_ids)])
    if missing:
        print("警告：這些要拿掉的名稱在資料集裡沒找到：", missing)
    print("保留的水果類別（", len(new_names), "）：", new_names)
    if len(new_names) != 21:
        print("警告：保留的類別不是 21 類，請確認上面的清單。")

    summary = {}
    class_boxes = {n: 0 for n in new_names}
    for split, aliases in SPLITS:
        image_dir = find_image_dir(args.root, aliases)
        if image_dir is None:
            print("找不到資料組：", split, "（略過）")
            continue
        label_dir = label_dir_of(image_dir)
        out_images = os.path.join(args.out, "images", split)
        out_labels = os.path.join(args.out, "labels", split)
        os.makedirs(out_images, exist_ok=True)
        os.makedirs(out_labels, exist_ok=True)
        stat = dict(images=0, kept=0, only_non_fruit=0, no_label=0, boxes_kept=0, boxes_removed=0, bad_lines=0)
        for name in sorted(os.listdir(image_dir)):
            if not name.endswith(IMG_EXT):
                continue
            stat["images"] += 1
            stem = os.path.splitext(name)[0]
            label_path = os.path.join(label_dir, stem + ".txt")
            if not os.path.exists(label_path):
                stat["no_label"] += 1
                continue
            kept_lines = []
            for line in open(label_path, encoding="utf-8").read().splitlines():
                parts = line.split()
                if len(parts) < 5:
                    if line.strip():
                        stat["bad_lines"] += 1
                    continue
                try:
                    old = int(float(parts[0]))
                except ValueError:
                    stat["bad_lines"] += 1
                    continue
                if old in drop_ids or old not in new_id:
                    stat["boxes_removed"] += 1
                    continue
                kept_lines.append(" ".join([str(new_id[old])] + parts[1:]))
                class_boxes[old_names[old]] += 1
                stat["boxes_kept"] += 1
            if not kept_lines:
                stat["only_non_fruit"] += 1
                continue
            place(os.path.join(image_dir, name), os.path.join(out_images, name))
            with open(os.path.join(out_labels, stem + ".txt"), "w", encoding="utf-8") as f:
                f.write("\n".join(kept_lines) + "\n")
            stat["kept"] += 1
        summary[split] = stat
        print("\n[%s] 原本 %d 張 → 保留 %d 張；只有蔬菜而排除 %d 張；沒有標註檔 %d 張；框：保留 %d、刪掉 %d；壞行 %d" % (
            split, stat["images"], stat["kept"], stat["only_non_fruit"], stat["no_label"], stat["boxes_kept"], stat["boxes_removed"], stat["bad_lines"]))

    if "train" not in summary or summary["train"]["kept"] == 0:
        raise SystemExit("train 資料組是空的，請檢查資料集資料夾排法。")
    out_yaml = os.path.join(args.out, "data_fruit.yaml")
    config = {"path": os.path.abspath(args.out), "train": "images/train", "val": "images/val",
              "names": {i: n for i, n in enumerate(new_names)}}
    if "test" in summary:
        config["test"] = "images/test"
    if "val" not in summary:
        raise SystemExit("找不到 val 資料組。")
    with open(out_yaml, "w", encoding="utf-8") as f:
        yaml.safe_dump(config, f, allow_unicode=True, sort_keys=False)
    if args.names_out:
        with open(args.names_out, "w", encoding="utf-8") as f:
            f.write("\n".join(new_names) + "\n")
    print("\n各水果在新資料集的框數（所有資料組合計）：")
    for n in new_names:
        print("  %-14s %7d" % (n, class_boxes[n]))
    print("\n已產生：", out_yaml)
    print("訓練時請用 --data", out_yaml)


if __name__ == "__main__":
    sys.exit(main())

Writing B/B0f_make_fruit_only.py


In [7]:
!rm -rf /content/dfv_fruit
!python B/B0f_make_fruit_only.py --root $(cat /content/b_root.txt) --out /content/dfv_fruit --names_out /content/drive/MyDrive/fruit_AB_out/b_fruit_classes.txt

資料集 data.yaml： /content/dfv16/AgricultureDataset/data.yaml
原本 30 類： ['Apple', 'Apricot', 'Aubegine', 'Banana', 'Bean', 'Carrot', 'Cherry', 'Courgette', 'DragonFruit', 'Durian', 'Fig', 'Grape', 'Lemon', 'Lime', 'Lychee', 'Mango', 'Orange', 'Peach', 'Pear', 'Pepper', 'Pineapple', 'Plum', 'Pomegranate', 'Quince', 'Radish', 'Rice', 'Salad', 'Strawberry', 'Tomato', 'Watermelon']
要拿掉的類別（ 9 ）： ['Aubegine', 'Bean', 'Carrot', 'Courgette', 'Pepper', 'Radish', 'Rice', 'Salad', 'Tomato']
保留的水果類別（ 21 ）： ['Apple', 'Apricot', 'Banana', 'Cherry', 'DragonFruit', 'Durian', 'Fig', 'Grape', 'Lemon', 'Lime', 'Lychee', 'Mango', 'Orange', 'Peach', 'Pear', 'Pineapple', 'Plum', 'Pomegranate', 'Quince', 'Strawberry', 'Watermelon']

[train] 原本 15944 張 → 保留 12952 張；只有蔬菜而排除 2992 張；沒有標註檔 0 張；框：保留 158026、刪掉 12933；壞行 0

[val] 原本 2582 張 → 保留 2144 張；只有蔬菜而排除 438 張；沒有標註檔 0 張；框：保留 27682、刪掉 1751；壞行 0

[test] 原本 1690 張 → 保留 1417 張；只有蔬菜而排除 273 張；沒有標註檔 0 張；框：保留 16496、刪掉 1116；壞行 0

各水果在新資料集的框數（所有資料組合計）：
  Apple            6599

## 步驟 6　把補強與新增的資料合併進來（B0i）
**事前**：Drive 的 `MyDrive/fruit_AB_out/` 裡只要有 `fruit26_extra_dataset_curated.zip` 這一個檔案。
**6-1** 解壓縮到 `/content/extra/`（輸出圖片數應約 **639**）。**6-2** 存程式。**6-3** 執行，輸出要看到「現在共 26 類」，**請整段貼給 Claude 確認，再開始訓練**。

In [8]:
!rm -rf /content/extra && mkdir -p /content/extra
!unzip -q -o /content/drive/MyDrive/fruit_AB_out/fruit26_extra_dataset_curated.zip -d /content/extra
!ls /content/extra/fruit26_extra
!echo "圖片數：" && find /content/extra/fruit26_extra/images -type f | wc -l

data_fruit.yaml  exclude.txt  images  labels  manifest.csv  README.txt
圖片數：
639


In [9]:
%%writefile B/B0i_merge_curated.py
# -*- coding: utf-8 -*-
"""B0i：把「我挑好的補訓資料」fruit26_extra_dataset_curated.zip 合併進 B0f 做好的 21 類水果資料集，變成 26 類。
用法：python B/B0i_merge_curated.py --base /content/dfv_fruit --curated /content/extra/fruit26_extra
  --base      B0f 做出來的資料集（要有 data_fruit.yaml，21 類）
  --curated   解壓縮後的 fruit26_extra 資料夾（裡面有 images、labels、data_fruit.yaml，26 類）
做的事：
  1. 檢查 26 類名單的前 21 類，和 base 的 21 類名稱與順序完全相同（不同就停止，避免編號錯位）。
  2. 把 curated 的圖片與標註複製進 base 的 train／val／test。
  3. 新增類別（編號 21～25）的 train 圖片太少，重複放入（檔名加 _r0、_r1…），讓每類的有效張數接近 --target_eff，最多 --max_repeat 次。
  4. 把 base 的 data_fruit.yaml 改成 26 類。
"""
import argparse
import os
import shutil
import sys
from collections import Counter

import yaml

IMG_EXT = (".jpg", ".jpeg", ".png", ".JPG", ".JPEG", ".PNG")


def place(src, dst):
    if os.path.exists(dst):
        return
    try:
        os.link(src, dst)
    except Exception:
        shutil.copy2(src, dst)


def names_of(path):
    names = yaml.safe_load(open(path, encoding="utf-8"))["names"]
    return [str(names[i]) for i in sorted(names)] if isinstance(names, dict) else [str(n) for n in names]


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--base", required=True)
    ap.add_argument("--curated", required=True)
    ap.add_argument("--target_eff", type=int, default=80, help="新增類別希望的有效訓練張數；0 = 不重複放入")
    ap.add_argument("--max_repeat", type=int, default=6)
    ap.add_argument("--names_out", default="")
    args = ap.parse_args()

    base_yaml = os.path.join(args.base, "data_fruit.yaml")
    cur_yaml = os.path.join(args.curated, "data_fruit.yaml")
    for p in (base_yaml, cur_yaml):
        if not os.path.exists(p):
            raise SystemExit("找不到：" + p)
    base_names, cur_names = names_of(base_yaml), names_of(cur_yaml)
    print("base：%d 類；curated：%d 類" % (len(base_names), len(cur_names)))
    if len(base_names) != 21:
        raise SystemExit("base 不是 21 類（可能已經合併過；請重新執行 B0f 再合併一次）：%d 類" % len(base_names))
    if cur_names[:21] != base_names:
        diff = [(i, a, b) for i, (a, b) in enumerate(zip(base_names, cur_names[:21])) if a != b]
        raise SystemExit("前 21 類的名稱或順序不一樣，不能合併（編號會錯位）：" + str(diff[:5]))
    new_ids = set(range(21, len(cur_names)))

    # 先數：新增類別在 train 的原圖張數
    per_class = Counter()
    train_items = []
    lab_dir = os.path.join(args.curated, "labels", "train")
    for f in sorted(os.listdir(lab_dir)) if os.path.isdir(lab_dir) else []:
        ids = {int(l.split()[0]) for l in open(os.path.join(lab_dir, f), encoding="utf-8").read().splitlines() if l.strip()}
        cls = ids & new_ids
        train_items.append((os.path.splitext(f)[0], cls))
        for c in cls:
            per_class[c] += 1
    repeat = {c: (1 if args.target_eff <= 0 else max(1, min(args.max_repeat, -(-args.target_eff // max(1, per_class[c]))))) for c in per_class}
    print("新增類別的訓練原圖張數與重複次數：", {cur_names[c]: (per_class[c], repeat[c]) for c in sorted(per_class)})

    stat = Counter()
    for split in ("train", "val", "test"):
        idir, ldir = os.path.join(args.curated, "images", split), os.path.join(args.curated, "labels", split)
        if not os.path.isdir(idir):
            continue
        oi, ol = os.path.join(args.base, "images", split), os.path.join(args.base, "labels", split)
        os.makedirs(oi, exist_ok=True)
        os.makedirs(ol, exist_ok=True)
        cls_of = dict(train_items) if split == "train" else {}
        for name in sorted(os.listdir(idir)):
            if not name.endswith(IMG_EXT):
                continue
            stem, ext = os.path.splitext(name)
            lp = os.path.join(ldir, stem + ".txt")
            if not os.path.exists(lp):
                stat["缺標註"] += 1
                continue
            copies = max([repeat[c] for c in cls_of.get(stem, set())] or [1]) if split == "train" else 1
            for k in range(copies):
                suffix = "_r%d" % k if copies > 1 else ""
                place(os.path.join(idir, name), os.path.join(oi, "x_" + stem + suffix + ext))
                shutil.copy2(lp, os.path.join(ol, "x_" + stem + suffix + ".txt"))
            stat[split + " 原圖"] += 1
            stat[split + " 放入的檔案"] += copies

    config = yaml.safe_load(open(base_yaml, encoding="utf-8"))
    config["names"] = {i: n for i, n in enumerate(cur_names)}
    with open(base_yaml, "w", encoding="utf-8") as f:
        yaml.safe_dump(config, f, allow_unicode=True, sort_keys=False)
    if args.names_out:
        with open(args.names_out, "w", encoding="utf-8") as f:
            f.write("\n".join(cur_names) + "\n")
    print("\n合併結果：", dict(stat))
    print("現在共 %d 類：%s" % (len(cur_names), cur_names))
    print("yaml：", base_yaml)


if __name__ == "__main__":
    sys.exit(main())


Writing B/B0i_merge_curated.py


In [10]:
!python B/B0i_merge_curated.py --base /content/dfv_fruit --curated /content/extra/fruit26_extra --names_out /content/drive/MyDrive/fruit_AB_out/b_fruit26_classes.txt
!cat /content/dfv_fruit/data_fruit.yaml

base：21 類；curated：26 類
新增類別的訓練原圖張數與重複次數： {'Kiwi': (16, 5), 'Raspberry': (15, 6), 'Blackberry': (22, 4), 'Cantaloupe': (30, 3), 'Jackfruit': (104, 1)}

合併結果： {'train 原圖': 453, 'train 放入的檔案': 697, 'val 原圖': 116, 'val 放入的檔案': 116, 'test 原圖': 70, 'test 放入的檔案': 70}
現在共 26 類：['Apple', 'Apricot', 'Banana', 'Cherry', 'DragonFruit', 'Durian', 'Fig', 'Grape', 'Lemon', 'Lime', 'Lychee', 'Mango', 'Orange', 'Peach', 'Pear', 'Pineapple', 'Plum', 'Pomegranate', 'Quince', 'Strawberry', 'Watermelon', 'Kiwi', 'Raspberry', 'Blackberry', 'Cantaloupe', 'Jackfruit']
yaml： /content/dfv_fruit/data_fruit.yaml
path: /content/dfv_fruit
train: images/train
val: images/val
names:
  0: Apple
  1: Apricot
  2: Banana
  3: Cherry
  4: DragonFruit
  5: Durian
  6: Fig
  7: Grape
  8: Lemon
  9: Lime
  10: Lychee
  11: Mango
  12: Orange
  13: Peach
  14: Pear
  15: Pineapple
  16: Plum
  17: Pomegranate
  18: Quince
  19: Strawberry
  20: Watermelon
  21: Kiwi
  22: Raspberry
  23: Blackberry
  24: Cantaloupe
  25: Ja

## 步驟 7　訓練 YOLO（26 類）
先刪掉前面失敗留下的空紀錄，再訓練。想先試跑：把 `EPOCHS = 50` 改成 `EPOCHS = 1`。
訓練完才會把 `best.pt` 複製到 Drive 的 `b_yolo_fruit26.pt`。

In [11]:
import os, shutil
from ultralytics import YOLO
EPOCHS = 50
assert os.path.exists('/content/dfv_fruit/data_fruit.yaml'), '找不到 data_fruit.yaml，請先完成步驟 5、6'
shutil.rmtree('/content/fruit_AB/runs/detect/b_yolo_fruit26', ignore_errors=True)
model = YOLO('yolov8s.pt')
model.train(data='/content/dfv_fruit/data_fruit.yaml', epochs=EPOCHS, patience=15, batch=16, imgsz=640, seed=0,
            project='/content/fruit_AB/runs/detect', name='b_yolo_fruit26', exist_ok=True)
best = '/content/fruit_AB/runs/detect/b_yolo_fruit26/weights/best.pt'
assert os.path.exists(best), '訓練沒有產生 best.pt'
shutil.copy(best, '/content/drive/MyDrive/fruit_AB_out/b_yolo_fruit26.pt')
print('已存到 Drive：b_yolo_fruit26.pt')

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
Ultralytics 8.4.202 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/dfv_fruit/data_fruit.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.

## 步驟 8　看新模型的成績
印出 val 與 test 的精確率、召回率、mAP，以及每個水果的 AP50。
**注意**：val、test 裡新增的 5 類圖片很少（每類只有幾張到十幾張），它們的 AP50 參考性很低；要看原本 21 類的 AP50 有沒有比舊模型低很多。真正的比較要等你用 189 張照片重跑。

In [12]:
from ultralytics import YOLO
import glob
paths = sorted(glob.glob('/content/fruit_AB/runs/detect/b_yolo_fruit26*/weights/best.pt'))
w = paths[-1]
print('權重：', w)
m = YOLO(w)
print('類別數：', len(m.names), list(m.names.values()))
for split in ('val', 'test'):
    r = m.val(data='/content/dfv_fruit/data_fruit.yaml', split=split, imgsz=640, plots=False)
    print('[%s] 精確率 %.3f  召回率 %.3f  mAP50 %.3f  mAP50-95 %.3f' % (split, r.box.mp, r.box.mr, r.box.map50, r.box.map))
    for i, ap in zip(r.box.ap_class_index, r.box.ap50):
        print('    %-14s AP50 %.3f' % (m.names[int(i)], float(ap)))

權重： /content/fruit_AB/runs/detect/b_yolo_fruit26/weights/best.pt
類別數： 26 ['Apple', 'Apricot', 'Banana', 'Cherry', 'DragonFruit', 'Durian', 'Fig', 'Grape', 'Lemon', 'Lime', 'Lychee', 'Mango', 'Orange', 'Peach', 'Pear', 'Pineapple', 'Plum', 'Pomegranate', 'Quince', 'Strawberry', 'Watermelon', 'Kiwi', 'Raspberry', 'Blackberry', 'Cantaloupe', 'Jackfruit']
Ultralytics 8.4.202 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 72 layers, 11,135,646 parameters, 0 gradients, 28.5 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1334.7±454.9 MB/s, size: 56.6 KB)
val: Scanning /content/dfv_fruit/labels/val.cache... 2260 images, 0 backgrounds, 14 corrupt: 100% ━━━━━━━━━━━━ 2260/2260 631.9Mit/s 0.0s
val: /content/dfv_fruit/images/val/Apple-tree-Pictures_jpg.rf.ac27870ad05fd02f8635c868dbdd64a5.jpg: ignoring corrupt image/label: labels mix segment and detection rows
val: /content/dfv_fruit/images/val/Apple-tree-Wallpaper_jpg.rf.794f0002f2f9e1c35e7318a7b767

## 步驟 9　備份到 Drive
訓練紀錄、曲線、混淆矩陣、yaml、類別清單都存起來。

In [13]:
OUT = "/content/drive/MyDrive/fruit_AB_out/b_fruit26"
!mkdir -p {OUT}
!cp -r /content/fruit_AB/runs/detect/b_yolo_fruit26* {OUT}/
!cp /content/dfv_fruit/data_fruit.yaml {OUT}/
!ls -R {OUT} | head -n 40

/content/drive/MyDrive/fruit_AB_out/b_fruit26:
b_yolo_fruit26
b_yolo_fruit26-2
data_fruit.yaml

/content/drive/MyDrive/fruit_AB_out/b_fruit26/b_yolo_fruit26:
args.yaml
BoxF1_curve.png
BoxP_curve.png
BoxPR_curve.png
BoxR_curve.png
confusion_matrix_normalized.png
confusion_matrix.png
labels.jpg
results.csv
results.png
train_batch0.jpg
train_batch1.jpg
train_batch2.jpg
train_batch34040.jpg
train_batch34041.jpg
train_batch34042.jpg
val_batch0_labels.jpg
val_batch0_pred.jpg
val_batch1_labels.jpg
val_batch1_pred.jpg
val_batch2_labels.jpg
val_batch2_pred.jpg
weights

/content/drive/MyDrive/fruit_AB_out/b_fruit26/b_yolo_fruit26/weights:
best.pt
last.pt

/content/drive/MyDrive/fruit_AB_out/b_fruit26/b_yolo_fruit26-2:
args.yaml
weights

/content/drive/MyDrive/fruit_AB_out/b_fruit26/b_yolo_fruit26-2/weights:


In [14]:
OUT = "/content/drive/MyDrive/fruit_AB_out/b_fruit26"
!mkdir -p {OUT}
# 備份所有 YOLO 訓練輸出的圖表、紀錄與權重
!cp -r /content/fruit_AB/runs/detect/b_yolo_fruit26* {OUT}/
# 備份 26 類的資料集設定 yaml
!cp /content/dfv_fruit/data_fruit.yaml {OUT}/
!ls -R {OUT} | head -n 40


/content/drive/MyDrive/fruit_AB_out/b_fruit26:
b_yolo_fruit26
b_yolo_fruit26-2
data_fruit.yaml

/content/drive/MyDrive/fruit_AB_out/b_fruit26/b_yolo_fruit26:
args.yaml
BoxF1_curve.png
BoxP_curve.png
BoxPR_curve.png
BoxR_curve.png
confusion_matrix_normalized.png
confusion_matrix.png
labels.jpg
results.csv
results.png
train_batch0.jpg
train_batch1.jpg
train_batch2.jpg
train_batch34040.jpg
train_batch34041.jpg
train_batch34042.jpg
val_batch0_labels.jpg
val_batch0_pred.jpg
val_batch1_labels.jpg
val_batch1_pred.jpg
val_batch2_labels.jpg
val_batch2_pred.jpg
weights

/content/drive/MyDrive/fruit_AB_out/b_fruit26/b_yolo_fruit26/weights:
best.pt
last.pt

/content/drive/MyDrive/fruit_AB_out/b_fruit26/b_yolo_fruit26-2:
args.yaml
weights

/content/drive/MyDrive/fruit_AB_out/b_fruit26/b_yolo_fruit26-2/weights:


## 步驟 10　把新權重放進網頁（**先不要放上 Hugging Face**）
1. 到 Drive 的 `fruit_AB_out` 下載 `b_yolo_fruit26.pt`。
2. 本機 `3_Local\models\` 裡先把舊的 `b_yolo.pt` 改名備份（例如 `b_yolo_30class.pt`），再把新檔改名成 `b_yolo.pt` 放進去。
3. `app.py` 的類別清單要跟著改（`B_CLASSES` 改成 26 類）：**新模型確認後再改，我會幫你改。**
4. 用 `start_research.bat`，設定全部維持 R0，上傳 `stock_test_set` 80 張與 `real_test109` 109 張，下載結果給我，和舊模型的 R0 比較。重點看：新增的 5 類有沒有被框到並叫對，**原本 21 類水果有沒有退步**。